# Ingestão e Limpeza: Produção do Feijão

Este notebook executa o pipeline de dados para extrair, limpar e estruturar os dados históricos de produção de feijão (em toneladas) por Unidade da Federação. Toda a lógica de dados está contida em `src/ingestion_producao_feijao.py`.

In [43]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Localiza a raiz do projeto
caminho_raiz = Path.cwd()
while not (caminho_raiz / "src").exists() and caminho_raiz.name != "":
    caminho_raiz = caminho_raiz.parent

# Adiciona ao path
if str(caminho_raiz) not in sys.path:
    sys.path.append(str(caminho_raiz))

print(f"Caminho raiz configurado para: {caminho_raiz}")

# Faz a importação
from src.ingestion_producao_feijao import processar_dados_producao_feijao

# Executa o pipeline
df_producao_feijao = processar_dados_producao_feijao()

# ==========================================
# VERIFICAÇÕES PARA GARANTIA DA MODELAGEM
# ==========================================
print("\n--- Informações do Dataset ---")
df_producao_feijao.info()

print("\n--- Amostra da Transição (Final de 2006 / Início de 2007) ---")
# Vamos filtrar um estado forte (ex: Paraná) bem na virada de 2006 para 2007
# para você ver que a base antiga (inferida) e a nova (matriz CONAB) se encaixaram
filtro_teste = df_producao_feijao[
    (df_producao_feijao['uf'] == 'parana') & 
    (df_producao_feijao['ano'].isin([2006, 2007]))
]
display(filtro_teste)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Caminho raiz configurado para: c:\Users\leand\Documents\mecai-26-prob-estat


c:\Users\leand\Documents\mecai-26-prob-estat\src\ingestion_producao_feijao.py:16: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  1: {1: 0.20, 2: 0.40, 3: 0.30, 4: 0.10},



--- Informações do Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 3910 entries, 0 to 3909
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ano              3910 non-null   int64  
 1   mes              3910 non-null   int64  
 2   uf               3910 non-null   str    
 3   regiao           3910 non-null   str    
 4   quantidade_tons  3910 non-null   float64
 5   is_estimado      3910 non-null   bool   
dtypes: bool(1), float64(1), int64(2), str(2)
memory usage: 223.2 KB

--- Amostra da Transição (Final de 2006 / Início de 2007) ---


,ano,mes,uf,regiao,quantidade_tons,is_estimado
324,2006,1,parana,Sul,180351.0,False
334,2006,2,parana,Sul,180351.0,False
340,2006,3,parana,Sul,45088.0,False
350,2006,4,parana,Sul,65634.0,False
367,2006,5,parana,Sul,131268.0,False
385,2006,6,parana,Sul,98451.0,False
399,2006,7,parana,Sul,32817.0,False
416,2006,8,parana,Sul,4571.0,False
437,2006,9,parana,Sul,4571.0,False
456,2006,10,parana,Sul,2285.0,False


In [44]:
df_producao_feijao.info()

<class 'pandas.DataFrame'>
RangeIndex: 3910 entries, 0 to 3909
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ano              3910 non-null   int64  
 1   mes              3910 non-null   int64  
 2   uf               3910 non-null   str    
 3   regiao           3910 non-null   str    
 4   quantidade_tons  3910 non-null   float64
 5   is_estimado      3910 non-null   bool   
dtypes: bool(1), float64(1), int64(2), str(2)
memory usage: 223.2 KB


In [45]:
# # 1. Agrupa pelas colunas desejadas e soma apenas a quantidade
# # 2. Usa o .reset_index() para transformar 'ano', 'mes' e 'uf' de volta em colunas da tabela
# producao_total = (
#     df_producao_feijao.groupby(["ano", "mes", "uf"], observed=True)["quantidade_tons"]
#     .sum()
#     .reset_index()
# )

# # 3. Opcional: Renomeia a coluna final para refletir que é a soma mensal
# producao_total = producao_total.rename(columns={"quantidade_tons": "soma_mensal_producao"})

# # 4. Ordena os dados (ex: do ano/mês mais recente para o mais antigo, ou por maior produção)
# producao_total = producao_total.sort_values(by=["uf","ano", "mes", "soma_mensal_producao"], ascending=[True, True, True, False])

# producao_total.rename(columns={"uf":"estado_origem"}, inplace=True)

# # 5. Salva no Excel mantendo as colunas visíveis
# producao_total.to_excel("s_producao_feijao_tons.xlsx", index=False)
# producao_total.to_parquet("s_producao_feijao_tons.parquet", index=False)


In [46]:
# producao_total

In [47]:
import pandas as pd
from pathlib import Path

# 1. Localiza a raiz do projeto de forma infalível
caminho_raiz = Path.cwd().resolve()
while not (caminho_raiz / "data").exists() and caminho_raiz.name != "":
    caminho_raiz = caminho_raiz.parent

caminho_final = caminho_raiz / "data" / "processed" / "consolidado" / "painel_feijao_consolidado.parquet"

try:
    df = pd.read_parquet(caminho_final)
    print("✅ Painel carregado com sucesso!\n")
    
    print("--- 1. TESTE DE DUPLICIDADE ---")
    dups = df.groupby(['data_referencia', 'uf']).size()
    print(f"Total de linhas duplicadas: {len(dups[dups > 1])} (Deve ser 0)")

    print("\n--- 2. TESTE DE TRANSIÇÃO (Fim de 2005 / Início de 2006) ---")
    mask_transicao = (df['ano'].isin([2005, 2006])) & (df['uf'] == 'SP')
    amostra = df[mask_transicao][['data_referencia', 'oferta_interna_estimada_ton', 'is_estimado']].drop_duplicates().sort_values('data_referencia')
    print(amostra[(amostra['data_referencia'] >= '2005-09-01') & (amostra['data_referencia'] <= '2006-03-01')])

    print("\n--- 3. TESTE DA ENTRESSAFRA (Julho de 2007) ---")
    mask_julho = (df['ano'] == 2007) & (df['mes'] == 7) & (df['uf'].isin(['SP', 'PR', 'SC']))
    print(df[mask_julho][['uf', 'oferta_interna_estimada_ton', 'is_estimado']].drop_duplicates())
    
except FileNotFoundError:
    print(f"❌ Erro: O arquivo não foi encontrado no caminho:\n{caminho_final}")
except Exception as e:
    print(f"❌ Ocorreu um erro inesperado: {e}")

✅ Painel carregado com sucesso!

--- 1. TESTE DE DUPLICIDADE ---
Total de linhas duplicadas: 0 (Deve ser 0)

--- 2. TESTE DE TRANSIÇÃO (Fim de 2005 / Início de 2006) ---
     data_referencia  oferta_interna_estimada_ton  is_estimado
1104      2005-09-01                      57521.0         True
1105      2005-10-01                      31020.0         True
1106      2005-11-01                      12004.0        False
1107      2005-12-01                      59036.0        False
1108      2006-01-01                     126817.0        False
1109      2006-02-01                     102862.0        False
1110      2006-03-01                      34099.0        False

--- 3. TESTE DA ENTRESSAFRA (Julho de 2007) ---
      uf  oferta_interna_estimada_ton  is_estimado
1126  SP                      43863.0        False
1668  PR                      16948.0        False
1939  SC                       6334.0        False
